# Notebook 11 Main · Exploratory report

CPU. Attach the exploratory Main Stage 05–08 releases and the corrected Stage 10 release. Attach the pilot Stage 03–09 releases matched by Stage 10 to reconstruct the sample provenance. Run all cells. The report gives score availability and paired descriptive means for all seven endpoints.


In [ ]:
# Notebook 11 · Cell 01 · Verify releases and registered inference helpers

from collections import Counter, defaultdict
from fractions import Fraction
import csv
import hashlib
import io
import json
import math
import os
from pathlib import Path, PurePosixPath
import platform
import random
import re
from statistics import NormalDist, mean, stdev
import types
import zipfile

import numpy as np
import scipy
from scipy.stats import norm, t
import yaml

INPUT = Path(os.environ.get("OPENAPI_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("OPENAPI_WORK", "/kaggle/working"))
STRATA = ("low", "medium", "high")
PILOT_VERSION = "stage08-fresh-pilot-v3-full-claim-review"
PILOT_HASHES = {
    "03": "fe26961c4fa5def49f7a3db13f2f9cd7f80e25f6717ab86a1e70f535b5d385fa",
    "04": "86ed5e55413d29d0a3d45b8270976424ddbbe74d48a0cfa9bca4be3992ba37d9",
    "05": "14698c5c34d1b693d062f993bb71d48b9f9b73d57d16c07613519dbda111ad0d",
    "06": "d47e519366bfe44ea9364f9539da838d221aff686215ddeb141eeb84f462b902",
    "07": "3da3c1a7cca4cdd344d6cbaa5f1564632dc453cb937fcd8e38ff935c1836b1bb",
    "08": "c3e71cfcb6d5ba071eab95fe159fb3a4a31a8184e4d3f38341292e45c3bb8136",
}
MAIN_PRODUCERS = {"09": "09_pilot_scoring", "10": "10_power_and_main_sampling",
                  "11": "11_main_inference_and_report"}


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha(data):
    return hashlib.sha256(data).hexdigest()


def file_hash(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def json_bytes(data):
    return (json.dumps(data, sort_keys=True, ensure_ascii=False, allow_nan=False,
                       separators=(",", ":")) + "\n").encode("utf-8")


def digest(data):
    return sha(json_bytes(data).rstrip(b"\n"))


def csv_rows(data):
    return list(csv.DictReader(io.StringIO(data.decode("utf-8-sig"))))


def jsonl_rows(data):
    return [json.loads(line) for line in data.splitlines() if line.strip()]


def csv_bytes(rows, fields=None):
    buffer = io.StringIO(newline="")
    writer = csv.DictWriter(buffer, fieldnames=fields or list(rows[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    return buffer.getvalue().encode("utf-8")


def keyed(rows, key, description):
    mapping = {key(row): row for row in rows}
    require(len(mapping) == len(rows), f"Duplicate identity in {description}")
    return mapping


def group(rows, key):
    result = defaultdict(list)
    for row in rows:
        result[key(row)].append(row)
    return result


def path_name(value):
    path = PurePosixPath(value)
    require(value not in ("", ".") and not path.is_absolute() and ".." not in path.parts
            and "\\" not in value and str(path) == value, f"Invalid artifact path {value!r}")
    return value


class Release:
    def __init__(self, stage, partition=None, pin=None):
        suffix = "_main" if partition == "main" and stage in {"05", "06", "07", "08"} else ""
        self.stage = stage
        self.archive = None
        names = sorted(INPUT.rglob(f"stage_{stage}{suffix}_manifest.json"))
        matches = []
        for path in names:
            data = path.read_bytes()
            meta = json.loads(data)
            if pin and sha(data) != pin:
                continue
            if partition and meta.get("partition") != partition:
                continue
            matches.append((path, data, meta))
        require(len(matches) == 1, f"Stage {stage} {partition or ''}: expected one matching manifest, found {len(matches)}")
        path, data, self.external = matches[0]
        require(self.external.get("stage") == stage and self.external.get("status") == "PASS",
                f"Stage {stage} input has not passed")
        self.external_hash = sha(data)
        description = self.external.get("archive")
        name = description["filename"] if isinstance(description, dict) else description
        name = name or self.external["archive_name"]
        require(Path(name).name == name, "Invalid archive name")
        archive = path.parent / name
        relative = f"registries/{stage}_manifest.json"
        self.internal_hash = self.external["internal_manifest_sha256"]
        if archive.is_file():
            require(file_hash(archive) == self.external["archive_sha256"], f"Stage {stage} archive hash differs")
            self.archive = zipfile.ZipFile(archive)
            names = self.archive.namelist()
            candidates = [n for n in names if n == relative or n.endswith("/" + relative)]
            require(len(set(names)) == len(names) and len(candidates) == 1, f"Stage {stage} archive layout differs")
            self.prefix = candidates[0][:-len(relative)]
            manifest_bytes = self.archive.read(candidates[0])
        else:
            candidates = [p for p in path.parent.rglob(relative)
                          if p.is_file() and file_hash(p) == self.internal_hash]
            require(len(candidates) == 1, f"Stage {stage} missing or ambiguous expanded manifest")
            self.root = candidates[0].parent.parent
            manifest_bytes = candidates[0].read_bytes()
        require(sha(manifest_bytes) == self.internal_hash, f"Stage {stage} internal manifest differs")
        self.internal = json.loads(manifest_bytes)
        require(self.internal["stage"] == stage and self.internal["status"] == "PASS", f"Stage {stage} internal status differs")
        if partition:
            require(self.internal["partition"] == partition, f"Stage {stage} partition differs")
        def artifact_name(row):
            identities = {row[key] for key in ("artifact_path", "relative_path", "path") if key in row}
            require(len(identities) == 1, f"Stage {stage} inconsistent artifact path")
            return path_name(identities.pop())
        self.artifacts = keyed(self.internal["artifacts"], artifact_name, f"Stage {stage} manifest")
        require(len(self.artifacts) == self.internal["artifact_count"], f"Stage {stage} artifact count differs")
        for n, record in self.artifacts.items():
            require(record["artifact_path"] == n and record["schema_version"] == "openapi-artifact-v1"
                    and bool(record["producer_notebook"])
                    and record["config_hash"] == self.internal["configuration_hash"]
                    and record["upstream_manifest_hash"] == digest(self.internal["upstream_manifest_sha256"]),
                    f"Stage {stage} artifact provenance differs: {n}")
        self.checked = set()

    def read(self, relative):
        record = self.artifacts[relative]
        data = self.archive.read(self.prefix + relative) if self.archive else (self.root / relative).read_bytes()
        require(len(data) == record["bytes"] and sha(data) == record["sha256"],
                f"Stage {self.stage}: artifact differs at {relative}")
        self.checked.add(relative)
        return data

    def csv(self, relative):
        return csv_rows(self.read(relative))

    def jsonl(self, relative):
        return jsonl_rows(self.read(relative))

    def json(self, relative):
        return json.loads(self.read(relative))

    def yaml(self, relative):
        return yaml.safe_load(self.read(relative))

    def link(self, other):
        require(self.internal["upstream_manifest_sha256"].get(other.stage) == other.internal_hash,
                f"Stage {self.stage} does not match the attached Stage {other.stage}")


def published_pilot(stage):
    # An amended Stage 08 must be explicitly pinned by its external manifest hash.
    pin = os.environ.get("OPENAPI_PILOT08_SHA256", PILOT_HASHES[stage]) if stage == "08" else PILOT_HASHES[stage]
    return Release(stage, "pilot" if stage in {"06", "07", "08"} else None, pin)


def verify_links(releases):
    for release in releases.values():
        for stage in release.internal["upstream_manifest_sha256"]:
            if stage in releases:
                release.link(releases[stage])


def load_module(name, data):
    # Load only registered research modules; the analysis never runs generated tests.
    module = types.ModuleType(name)
    exec(compile(data, f"<registered {name}>", "exec", dont_inherit=True), module.__dict__)
    return module


def runtime():
    return {"python": platform.python_version(), "numpy": np.__version__,
            "scipy": scipy.__version__, "pyyaml": yaml.__version__}


def write_release(stage, partition, files, sources, metadata):
    destination = WORK / f"stage_{stage}_{partition}_dataset"
    destination.mkdir(parents=True, exist_ok=True)
    config_hash = digest({n: sha(b) for n, b in files.items() if n.startswith(("config/", "src/"))})
    upstream = {s: release.internal_hash for s, release in sources.items()}
    provenance = {"schema_version": "openapi-artifact-v1", "producer_notebook": MAIN_PRODUCERS[stage],
                  "config_hash": config_hash, "upstream_manifest_hash": digest(upstream)}
    artifacts = [{"artifact_path": n, "relative_path": n, "bytes": len(files[n]),
                  "sha256": sha(files[n]), **provenance} for n in sorted(files)]
    base = {"stage": stage, "status": "PASS", "partition": partition,
            "design_id": sources["06"].internal["design_id"], **metadata}
    internal = json_bytes({**base, "artifacts": artifacts, "artifact_count": len(artifacts),
                           "artifact_record_schema": "openapi-artifact-v1",
                           "configuration_hash": config_hash, "upstream_manifest_sha256": upstream})
    filename = f"stage_{stage}_{partition}.zip"
    prefix = f"stage_{stage}_{partition}/prose_operation_experiment"
    internal_path = f"registries/{stage}_manifest.json"
    tmp = destination / (filename + ".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=7) as archive:
        for n, b in sorted({**files, internal_path: internal}.items()):
            entry = zipfile.ZipInfo(prefix + "/" + n, (2026, 11, 11, 0, 0, 0))
            archive.writestr(entry, b, zipfile.ZIP_DEFLATED, 7)
    with zipfile.ZipFile(tmp) as archive:
        require(archive.testzip() is None and len(archive.namelist()) == len(artifacts) + 1
                and archive.read(prefix + "/" + internal_path) == internal
                and all(sha(archive.read(prefix + "/" + a["artifact_path"])) == a["sha256"] for a in artifacts),
                "Release archive verification failed")
    archive_path = destination / filename
    archive_hash = file_hash(tmp)
    manifest = json_bytes({**base, "archive": {"filename": filename, "project_prefix": prefix,
                                               "sha256": archive_hash, "bytes": tmp.stat().st_size,
                                               "member_count": len(artifacts) + 1},
                           "archive_sha256": archive_hash, "internal_manifest_sha256": sha(internal),
                           "internal_manifest": {"relative_path": internal_path,
                                                 "artifact_count": len(artifacts), "sha256": sha(internal)}})
    manifest_name = f"stage_{stage}_manifest.json"
    proposed = {manifest_name: manifest, f"stage_{stage}_checksums.sha256": (
        f"{archive_hash}  {filename}\n{sha(manifest)}  {manifest_name}\n").encode()}
    try:
        require(not archive_path.exists() or file_hash(archive_path) == archive_hash,
                f"Existing release differs at {archive_path}. Use a new OPENAPI_WORK directory.")
        for n, b in proposed.items():
            require(not (destination / n).exists() or (destination / n).read_bytes() == b,
                    f"Existing release differs at {destination / n}")
        tmp.replace(archive_path)
        for n, b in proposed.items():
            (destination / n).write_bytes(b)
    finally:
        tmp.unlink(missing_ok=True)
    print(f"Stage {stage} {partition} release verified at {destination}")
    return destination


def require_math(actual, expected, message):
    require(math.isclose(float(actual), float(expected), rel_tol=1e-9, abs_tol=1e-12), message)


def hamilton(sizes, n):
    require(0 <= n <= sum(sizes.values()), "Requested sample exceeds eligible remainder")
    quotas = {h: Fraction(n * sizes[h], sum(sizes.values())) for h in STRATA}
    counts = {h: int(quotas[h]) for h in STRATA}
    ranked = sorted(STRATA, key=lambda h: (-(quotas[h] - counts[h]), STRATA.index(h)))
    for h in ranked[:n - sum(counts.values())]:
        counts[h] += 1
    return counts


def main_frame(sources):
    s03, s04, s05 = (sources[s] for s in ("03", "04", "05"))
    frame = keyed(s03.csv("data/inventory/eligible_operations.csv"), lambda r: r["op_id"], "Stage 03 eligible frame")
    ranked = keyed(s04.jsonl("partitions/pilot/ranked_frame.jsonl"), lambda r: r["op_id"], "Stage 04 ranked frame")
    pilots = keyed(s05.jsonl("partitions/pilot/final_selected_operations.jsonl"), lambda r: r["op_id"], "pilot selection")
    exposed = {r["op_id"] for r in s03.csv("data/audits/development_exposure.csv")
               if r["exclusion_decision"] == "EXCLUDE_DEVELOPMENT_EXPOSED"}
    excluded = {r["op_id"] for r in s05.jsonl("reports/audit/frame_eligibility_exclusions.jsonl")}
    require(frame.keys() == ranked.keys() and pilots.keys() <= frame.keys()
            and not pilots.keys() & (exposed | excluded), "Selection provenance differs")
    design = s04.json("config/sampling_design.json")
    c1, c2 = (design["boundaries"][k] for k in ("c1", "c2"))
    for op, row in frame.items():
        tokens = int(row["prose_tokens_L0"])
        h = "low" if tokens <= c1 else "medium" if tokens <= c2 else "high"
        audit = ranked[op]
        require(audit["stratum"] == h and all(audit[k] == row[k] for k in
                ("op_id", "pair_id", "family_id", "spec_id", "L0_sha256", "L1_sha256"))
                and int(audit["prose_tokens_L0"]) == tokens, "Frozen frame row differs")
        require(row["final_status"] == "ELIGIBLE" and row["development_exposed"].lower() == "false",
                "Ineligible operation in Stage 03 frame")
    original_sizes = Counter(r["stratum"] for op, r in ranked.items() if op not in excluded)
    for row in s05.jsonl("partitions/pilot/final_stratum_sizes.jsonl"):
        require(original_sizes[row["stratum"]] == row["N_h"], "Stage 05 final population differs")
    reserve = [dict(frame[op], stratum=ranked[op]["stratum"]) for op in sorted(frame)
               if op not in pilots and op not in exposed and op not in excluded]
    sizes = {h: sum(r["stratum"] == h for r in reserve) for h in STRATA}
    require(all(sizes.values()), "A planned population stratum is empty")
    return reserve, sizes, pilots, {"c1": c1, "c2": c2,
                                   "stage05_excluded_operation_count": len(excluded),
                                   "stage03_development_excluded_operation_count": len(exposed)}


def input_file(name, required=True):
    matches = sorted(INPUT.rglob(name))
    require(len(matches) <= 1 and (not required or matches), f"Expected one input file named {name}")
    return matches[0].read_bytes() if matches else None


def check_registered_runtime(stage):
    registered = stage.yaml("config/analysis_registered.yaml")
    versions = stage.json("config/stage06_environment.json")["packages"]
    import statsmodels
    from statsmodels.api import WLS
    from statsmodels.stats.multitest import multipletests

    actual = {**runtime(), "statsmodels": statsmodels.__version__}
    for key in ("scipy", "numpy", "statsmodels"):
        require(actual[key] == versions[key], f"Registered {key} version is {versions[key]}, found {actual[key]}")
    require(actual["statsmodels"] == registered["maintained_package_version"], "Analysis package registration differs")
    return WLS, multipletests, actual


def validate_planning(plan, targets, endpoints, sizes):
    require(set(targets) == set(endpoints) and all(0 < v["target"] <= 1 and v["rationale"].strip()
            for v in targets.values()), "Provide justified positive effect targets for all seven endpoints")
    require(plan["power_goal"] in (0.8, 0.9) and str(plan["power_rationale"]).strip(),
            "Choose and justify power of 0.80 or 0.90")
    require(plan["pilot_effect_target_timing"] in {"BEFORE_INSPECTING_PILOT_DIFFERENCES",
            "AFTER_INSPECTING_PILOT_DIFFERENCES"} and str(plan["timing_note"]).strip(),
            "Record when meaningful targets were chosen")
    require(set(plan["seeds"]) == {"selection", "simulation"} and
            all(type(x) is int and x >= 0 for x in plan["seeds"].values()) and
            len(set(plan["seeds"].values()) | {20261103, 20261104, 20261105}) == 5,
            "Use independent integer selection/simulation seeds, distinct from frozen pilot seeds")
    require(type(plan["qualify_supplemental_bootstrap"]) is bool, "Record whether to qualify the supplemental bootstrap")
    require(0.90 <= plan["fwer_mc_confidence"] < 1 and plan["fwer_upper_limit"] == 0.05,
            "Predeclare a one-sided FWER confidence level of at least 90% and retain alpha .05")
    n_values = plan["candidate_sizes"]
    require(type(plan["max_pairs"]) is int and plan["max_pairs"] > 0 and len(set(n_values)) >= 2
            and all(type(n) is int and 0 < n <= min(sum(sizes.values()), plan["max_pairs"])
            and all(hamilton(sizes, n).values()) for n in n_values),
            "Supply at least two feasible candidate sizes with all strata represented")
    require(str(plan["budget_rationale"]).strip()
            and type(plan["simulation_iterations"]) is int and plan["simulation_iterations"] >= 1000,
            "Document the feasible budget and simulation count")
    require(0 < plan["minimum_pointwise_coverage"] <= .95 and str(plan["qualification_rationale"]).strip(),
            "Predeclare and justify the interval qualification rule")
    require(len(plan["scenarios"]) >= 2 and len({s["id"] for s in plan["scenarios"]}) == len(plan["scenarios"]),
            "Supply several justified nuisance scenarios")
    require(len({(s["family_latent_correlation"], s["endpoint_latent_correlation"])
                 for s in plan["scenarios"]}) >= 2, "Vary plausible family or endpoint dependence")
    for s in plan["scenarios"]:
        require(str(s["rationale"]).strip() and 0 <= s["family_latent_correlation"] < 1
                and 0 <= s["endpoint_latent_correlation"] < 1 and set(s["endpoints"]) == set(endpoints),
                "Invalid scenario assumptions")
        for ep, mapping in s["endpoints"].items():
            require(set(mapping) == set(STRATA), f"Missing nuisance stratum for {ep}")
            for config in mapping.values():
                require(0 < config["availability"] <= 1, "Invalid common-evaluable probability")
                for effect in (0.0, targets[ep]["target"]):
                    probabilities(ep, config, effect)


BINARY = {"collection_success", "baseline_pass"}


def probabilities(endpoint, cfg, delta):
    amplitude = 1.0 if endpoint in BINARY else float(cfg["amplitude"])
    require(0 < amplitude <= 1 and abs(delta) <= amplitude, "Unattainable contrast")
    q = float(cfg["discordance"]) if endpoint in BINARY else (float(cfg["sd_delta"])**2 + delta**2) / amplitude**2
    require(0 < q <= 1 and q >= abs(delta) / amplitude - 1e-12, "Incoherent paired outcome probabilities")
    negative = max(0., (q - delta/amplitude)/2)
    positive = max(0., (q + delta/amplitude)/2)
    if endpoint in BINARY:
        both_one = float(cfg["L0_probability"]) - negative
        require(-1e-12 <= both_one <= 1-q+1e-12, "Incoherent baseline marginal")
    return negative, positive, amplitude


def preliminary_numbers(plan, targets, endpoints):
    z_alpha = NormalDist().inv_cdf(1 - .05 / (2*7))
    z_power = NormalDist().inv_cdf(plan["power_goal"])
    estimates = []
    for scenario in plan["scenarios"]:
        for ep in endpoints:
            delta = targets[ep]["target"]
            for h, cfg in scenario["endpoints"][ep].items():
                if ep in BINARY:
                    q = cfg["discordance"]
                    n = (z_alpha*math.sqrt(q) + z_power*math.sqrt(q-delta**2))**2 / delta**2
                else:
                    n = (z_alpha+z_power)**2 * cfg["sd_delta"]**2 / delta**2
                estimates.append({"scenario": scenario["id"], "endpoint": ep, "stratum": h,
                                  "evaluable_pairs_approx": math.ceil(n),
                                  "sampled_pairs_approx": math.ceil(n/cfg["availability"]),
                                  "planning_approximation_only": True})
    return estimates


def sample_rows(reserve, allocation, rng):
    pools = group(reserve, lambda r: r["stratum"])
    return [pools[h][int(i)] for h in STRATA
            for i in rng.choice(len(pools[h]), size=allocation[h], replace=False)]


def cluster_estimate(statistics, WLS, rows, shares, conditional, actual_fit=True):
    if not rows or len({r["family_id"] for r in rows}) <= 1:
        raise ValueError("Insufficient evaluable families")
    if {r["stratum"] for r in rows} != set(STRATA):
        raise ValueError("A registered target stratum has no evaluable pair")
    result = statistics.primary_t_inference(rows, shares, conditional=conditional)
    if actual_fit:
        x = np.asarray([[r["stratum"] == h for h in STRATA] for r in rows], dtype=float)
        weights = np.asarray([1/r["pi"] for r in rows])
        y = np.asarray([r["d"] for r in rows])
        families = np.unique([r["family_id"] for r in rows], return_inverse=True)[1]
        w = np.asarray([result["target_shares"][h] for h in STRATA])
        fit = WLS(y, x, weights=weights, hasconst=False).fit(
            cov_type="cluster", cov_kwds={"groups": families, "use_correction": True,
                                         "df_correction": True}, use_t=True)
        require_math(float(w @ fit.params), result["effect"], "WLS effect differs from registered implementation")
        require_math(float(np.sqrt(w @ fit.cov_params() @ w)), result["standard_error"],
                     "Cluster CR1 covariance differs from registered implementation")
    return result


def wilson(k, n):
    if not n:
        return None, None
    z = 1.95996398454
    q = k/n
    d = 1 + z*z/n
    m = (q + z*z/(2*n))/d
    r = z*math.sqrt(q*(1-q)/n + z*z/(4*n*n))/d
    return max(0., m-r), min(1., m+r)


def multiplier_bootstrap(rows, target_shares, conditional, rng, draws):
    strata = {h: i for i, h in enumerate(STRATA)}
    families = {g: i for i, g in enumerate(sorted({r["family_id"] for r in rows}))}
    numerators = np.zeros((len(families), len(STRATA)))
    denominators = np.zeros_like(numerators)
    for row in rows:
        g, h = families[row["family_id"]], strata[row["stratum"]]
        weight = 1/row["pi"]
        numerators[g,h] += weight*row["d"]
        denominators[g,h] += weight
    fixed_target = np.asarray([target_shares[h] for h in STRATA])
    values = []
    for start in range(0, draws, 512):
        multipliers = rng.exponential(size=(min(512, draws-start), len(families)))
        numerator = multipliers @ numerators
        denominator = multipliers @ denominators
        effects = (numerator.sum(axis=1)/denominator.sum(axis=1) if conditional
                   else (numerator/denominator) @ fixed_target)
        values.extend(effects)
    return tuple(float(x) for x in np.quantile(values, (.025, .975)))


In [ ]:
# Notebook 11 · Cell 02 · Reconcile main execution and apply the same scoring code

EVIDENCE = "data/evidence/"

pilot_sources = {s: published_pilot(s) for s in ("03", "04", "05", "06", "07", "08")}
pilot_sources["09"] = Release("09", "pilot")
verify_links(pilot_sources)
plan_release = Release("10", "main")
require(plan_release.internal.get("checkpoint_d_reached") is True
        and plan_release.internal.get("may_begin_notebook_05_main") is True
        and plan_release.internal.get("main_generation_authorized") is False,
        "Attach a qualified Checkpoint D release")
for stage, release in pilot_sources.items():
    if stage != "07":
        plan_release.link(release)
plan = plan_release.yaml("partitions/main/main_plan.yaml")
analysis = plan_release.yaml("partitions/main/main_analysis_registered.yaml")
require(analysis.get("analysis_plan_version") == "stage10-main-v4-power70-conservative-holm"
        and analysis.get("power_goal_authority") == "SUPERVISOR_APPROVED_PROTOCOL_AMENDMENT"
        and math.isclose(analysis.get("familywise_alpha", 0), plan.get("analysis_decision_alpha", -1))
        and math.isclose(analysis.get("nominal_familywise_alpha", 0), .05)
        and math.isclose(analysis.get("power_target", 0), .70),
        "Attach the Stage 10 V4.1 qualified main analysis plan")
require(plan["qualification"]["fwer_qualified"] and plan["qualification"]["coverage_qualified"]
        and plan["qualification"]["power_goal_met"], "Main analysis qualification is incomplete")
fw = plan["qualification"].get("fwer_mc_acceptance", {})
require(fw.get("method") == "one-sided exact lower-bound falsification check per null scenario"
        and .90 <= fw.get("confidence", 0) < 1 and fw.get("registered_alpha") == .05
        and 0 < fw.get("analysis_decision_alpha", 1) <= fw.get("registered_alpha")
        and plan["qualification"].get("highest_null_fwer_mc_lower", 1) <= .05,
        "Stage 10 lacks a passing Monte Carlo FWER qualification")
selection = keyed(plan_release.csv("partitions/main/selected.csv"), lambda r: r["op_id"], "Stage 10 main selection")
pilot_selection = keyed(pilot_sources["05"].jsonl("partitions/pilot/final_selected_operations.jsonl"),
                        lambda r: r["op_id"], "pilot selection")
reserve, remainder_counts, _, frame_audit = main_frame(pilot_sources)
require(plan["N_h"] == remainder_counts and plan["frozen_prose_boundaries"] == frame_audit
        and hamilton(remainder_counts, plan["size"]) == plan["n_h"],
        "Main population or allocation differs from the frozen remainder")
regenerated = sample_rows(reserve, plan["n_h"],
                          np.random.Generator(np.random.PCG64(plan["selection_seed"])))
require({r["op_id"] for r in regenerated} == selection.keys(),
        "Selected identities do not reproduce from the frozen seed and frame")
for row in regenerated:
    selected = selection[row["op_id"]]
    require(all(selected[k] == row[k] for k in
                ("pair_id", "family_id", "spec_id", "L0_sha256", "L1_sha256", "stratum", "prose_tokens_L0")),
            "Selected row differs from the eligible frame")
require(len(selection) == plan["size"] and not selection.keys() & pilot_selection.keys(),
        "Main size or pilot exclusion differs")
require(Counter(r["stratum"] for r in selection.values()) == plan["n_h"], "Main stratum allocations differ")
require_math(sum(plan["W_h"].values()), 1, "Target stratum shares differ")
for row in selection.values():
    h = row["stratum"]
    require_math(row["inclusion_probability"], plan["n_h"][h]/plan["N_h"][h], "Main inclusion probability differs")
    require_math(plan["W_h"][h], plan["N_h"][h]/sum(plan["N_h"].values()), "Main stratum share differs")
    require(row["development_exposed"].lower() == "false" and row["final_status"] == "ELIGIBLE",
            "Ineligible main operation")
sources = {s: Release(s, "main") for s in ("05","06","07","08")}
require(sources["08"].external.get("may_begin_notebook_11") is True
        and sources["08"].external.get("review_protocol_version") == "stage08-main-v4-complete-accounting",
        "Attach the full-review main Stage 08 release")
verify_links(sources)
for release, upstream in (("05","10"),("06","10"),("06","05"),("07","06"),
                          ("08","05"),("08","06"),("08","07")):
    sources[release].link(plan_release if upstream == "10" else sources[upstream])
for release in sources.values():
    require(release.internal["design_id"] == pilot_sources["06"].internal["design_id"]
            and int(release.internal["operation_count"]) == len(selection), "Main design or operation count differs")
SCOPE = 'main-seven-endpoint-exploratory-v1'
require(all(release.internal.get("protocol_scope") == SCOPE for release in sources.values())
        and sources["05"].internal.get("technical_oracle_qualification_complete") is True
        and sources["05"].internal.get("human_review_complete") is False
        and sources["06"].internal.get("checkpoint_b_inherited") is False
        and sources["06"].json("reports/audit/stage_06_final_summary.json").get("all_evaluator_qualifications_passed") is True,
        "Missing matching exploratory Main releases")
oracle_selection = keyed(sources["05"].jsonl("partitions/main/final_selected_operations.jsonl"),
                         lambda r: r["op_id"], "main oracle review selection")
require(oracle_selection.keys() == selection.keys(), "Reviewed main oracles differ from Checkpoint D")
for op, row in selection.items():
    for key in ("pair_id", "family_id", "stratum", "L0_hash", "L1_hash"):
        require(row[key] == oracle_selection[op][key], "Main reviewed input differs from frozen selection")
    require_math(row["inclusion_probability"], oracle_selection[op]["inclusion_probability"],
                 "Main selection probability was changed")
for name in ("config/system_prompt.txt", "config/user_prompt.txt",
             "src/statistics.py"):
    require(sources["06"].read(name) == pilot_sources["06"].read(name),
            f"Main release contains an unqualified evaluator-contract change: {name}")
require(sources["06"].read("config/analysis_registered.yaml")
        == plan_release.read("partitions/main/main_analysis_registered.yaml"),
        "Main evaluator did not inherit the qualified Stage 10 analysis plan")
partial_path = "config/main_partial_evidence_qualification.json"
partial_raw = sources["06"].read(partial_path)
require(sha(partial_raw) == sources["06"].internal["main_partial_qualification_sha256"],
        "Stage 06 exploratory policy hash differs.")
partial_qualification = json.loads(partial_raw)
require(partial_qualification["status"] == "EXPLORATORY"
        and partial_qualification["protocol_scope"] == SCOPE
        and partial_qualification["stage10_internal_sha256"] == plan_release.internal_hash
        and partial_qualification["oracle_release_sha256"] == sources["05"].internal_hash
        and partial_qualification["review_csv_sha256"] == sources["05"].internal["review_csv_sha256"]
        and partial_qualification["operation_count"] == len(selection)
        and partial_qualification["confirmatory_endpoints"] == [],
        "Main exploratory analysis scope differs.")
for component in ("metrics", "worker", "routing", "statistics", "scoring_adapter"):
    require(sha(sources["06"].read(f"src/{component}.py")) == partial_qualification[f"{component}_sha256"],
            f"Qualified Main {component} implementation differs.")
adapter_qualification = sources["06"].json(
    "reports/qualification/main_scoring_adapter_qualification.json")
adapter = sources["06"].read("src/scoring_adapter.py")
require(sources["06"].internal.get("main_evaluator_version")
        == 'stage06-main-v7-seven-endpoint-exploratory'
        and adapter_qualification.get("status") == "PASS"
        and adapter_qualification.get("metric_definitions_changed") is False
        and adapter_qualification.get("denominators_changed") is True
        and adapter_qualification.get("missing_evidence_policy_changed") is True
        and adapter_qualification.get("pilot_adapter_sha256") == analysis["score_adapter_sha256"]
        and adapter_qualification.get("main_adapter_sha256") == sha(adapter),
        "Main scoring adapter was not compatibly versioned and requalified")
gen_main = sources["07"].json("config/generation_frozen.json")
gen_pilot = pilot_sources["07"].json("config/generation_frozen.json")
# GRANITE RUN: model_id, model_revision and tokenizer_revision differ from the Qwen pilot by design.
generation_keys = ("decoding", "dtype", "generation", "max_new_tokens")  # GRANITE RUN: input/total token budgets are tokenizer-specific
require(all(gen_main[k] == gen_pilot[k] for k in generation_keys)
        and sources["07"].read("config/extraction_policy.json") == pilot_sources["07"].read("config/extraction_policy.json"),
        "Main generation or extraction settings differ from the frozen pilot contract")
for name in ("src/worker.py", "src/routing.py", "src/metrics.py"):
    require(sources["08"].read(name) == sources["06"].read(name), f"Main evaluator differs: {name}")

package_runtime = runtime
require(sha(plan_release.read("src/statistics.py")) == analysis["implementation_sha256"]
        and plan_release.read("src/statistics.py") == sources["06"].read("src/statistics.py"),
        "Archived Stage 10 statistical implementation differs")
scorer = load_module("reused_scoring_adapter", adapter)
for name in ("require", "keyed", "group", "random", "re", "load_module"):
    scorer.__dict__[name] = globals()[name]
result = scorer.score_partition(sources, selection, "main", sources["06"].read("src/metrics.py"))
endpoints = tuple(result["endpoints"])
require(len(endpoints) == 7 and len(set(endpoints)) == 7,
        "Exactly seven unique primary endpoints are required")
expected_long_keys = {(op, condition, endpoint)
                      for op in selection for condition in ("L0", "L1") for endpoint in endpoints}
actual_long_keys = {(row["op_id"], row["condition"], row["endpoint"])
                    for row in result["long"]}
require(len(result["long"]) == len(expected_long_keys)
        and len(actual_long_keys) == len(expected_long_keys)
        and actual_long_keys == expected_long_keys,
        "Every operation-condition must have exactly one record for each of seven endpoints")
task_lookup = result["tasks"]
metric_ids = set()
for row in result["long"]:
    task = task_lookup[row["task_id"]]
    require(row["op_id"] == task["op_id"] and row["pair_id"] == task["pair_id"]
            and row["condition"] == task["condition"],
            "Metric record identity differs from the frozen task")
    metric_id = "metric_" + sha(json_bytes([row["task_id"], row["endpoint"]]))[:24]
    require(metric_id not in metric_ids, "Duplicate metric record identity")
    metric_ids.add(metric_id)
    row.update(metric_record_id=metric_id,
               evaluation_status="OBSERVED" if row["observed"] else "UNAVAILABLE",
               evidence_reference=("data/evidence/execution.csv;data/evidence/baseline_results.jsonl;"
                                   "data/evidence/request_attempts.jsonl;data/evidence/assertion_claims.jsonl;"
                                   "data/evidence/opportunity_outcomes.jsonl"))
    require((row["score"] is not None) == bool(row["observed"]),
            "Observed flag and metric score availability disagree")
    require(row["observed"] or bool(row["failure_reason"]),
            "Unavailable metric record lacks an explicit reason")
expected_pair_keys = {(op, endpoint) for op in selection for endpoint in endpoints}
actual_pair_keys = {(row["op_id"], row["endpoint"]) for row in result["paired"]}
require(len(result["paired"]) == len(expected_pair_keys)
        and len(actual_pair_keys) == len(expected_pair_keys)
        and actual_pair_keys == expected_pair_keys,
        "Every operation must have exactly one paired record for each of seven endpoints")
paired_ids = set()
for row in result["paired"]:
    pair_id = "paired_metric_" + sha(json_bytes([row["op_id"], row["endpoint"]]))[:24]
    require(pair_id not in paired_ids, "Duplicate paired metric record identity")
    paired_ids.add(pair_id)
    row.update(paired_metric_record_id=pair_id,
               pair_status="COMMON_EVALUABLE" if row["common_evaluable"] else "UNAVAILABLE")
    require(row["common_evaluable"] or row["L0_unavailable_reason"] or row["L1_unavailable_reason"],
            "Unavailable pair lacks an explicit condition-level reason")
metric_accounting = []
for condition in ("L0", "L1"):
    for endpoint in endpoints:
        rows = [row for row in result["long"]
                if row["condition"] == condition and row["endpoint"] == endpoint]
        metric_accounting.append({
            "condition": condition, "endpoint": endpoint,
            "expected_records": len(selection), "actual_records": len(rows),
            "observed_records": sum(row["observed"] for row in rows),
            "unavailable_records": sum(not row["observed"] for row in rows),
            "status": "COMPLETE" if len(rows) == len(selection) else "INCOMPLETE",
        })
require(all(row["status"] == "COMPLETE" for row in metric_accounting),
        "Metric accounting summary is incomplete")
print("Main task, review, opportunity and score ledgers verified", len(selection),
      "operations", len(result["long"]), "task-endpoint records", len(result["paired"]),
      "paired-endpoint records")


In [ ]:
# Notebook 11 · Cell 03 · Calculate effects, intervals, Holm adjustment and diagnostics

shares = plan["W_h"]
primary = []
for ep, all_rows in group(result["paired"], lambda r: r["endpoint"]).items():
    usable = [r for r in all_rows if r["common_evaluable"]]
    descriptive = {key: mean(float(r[key]) for r in usable) if usable else None
                   for key in ("L0", "L1", "delta")}
    primary.append({"endpoint": ep, "total_pairs": len(all_rows),
                    "common_evaluable_pairs": len(usable),
                    "missing_pairs": len(all_rows) - len(usable),
                    "missing_L0": sum(r["L0"] is None for r in all_rows),
                    "missing_L1": sum(r["L1"] is None for r in all_rows),
                    "observed_pairs_unweighted_L0": descriptive["L0"],
                    "observed_pairs_unweighted_L1": descriptive["L1"],
                    "observed_pairs_unweighted_L1_minus_L0": descriptive["delta"],
                    "inference_status": "DESCRIPTIVE_ONLY" if usable else "NOT_ESTIMABLE",
                    "effect_L1_minus_L0": None, "pointwise_ci_low": None, "pointwise_ci_high": None})
require(len(primary) == 7, "Report must account for all seven metrics")

tasks, artifacts, exe = (result[k] for k in ("tasks","artifacts","execution"))
tokens = []
for op, pair in sorted(group(tasks.values(), lambda r:r["op_id"]).items()):
    a,b = sorted(pair,key=lambda r:r["condition"])
    x,y = int(a["prompt_tokens"]),int(b["prompt_tokens"])
    require(x <= 5376 and y <= 5376 and x > 0 and y > 0, "Expected untruncated input within frozen token limit")
    f0,f1 = (artifacts[r["task_id"]]["finish_reason"] for r in (a,b))
    tokens.append({"op_id":op,"stratum":selection[op]["stratum"],"L0_prompt_tokens":x,"L1_prompt_tokens":y,
                   "delta_tokens":y-x,"relative_change":(y-x)/x,
                   "L0_finish_reason":f0,"L1_finish_reason":f1,
                   "L0_generation_cap_reached":f0=="max_new_tokens","L1_generation_cap_reached":f1=="max_new_tokens"})
outcomes, diagnostics = [], []
for cond in ("L0","L1"):
    keys = {k for k,v in tasks.items() if v["condition"]==cond}
    er = [exe[k] for k in keys]
    outcomes.append({"condition":cond,"expected":len(keys),"raw":sum(bool(artifacts[k]["raw_hash"]) for k in keys),
                     "extractable":sum(bool(artifacts[k]["extracted_hash"]) for k in keys),
                     "compiled":sum(r["compile_status"]=="PASS" for r in er),
                     "collected":sum(r["collection_status"]=="PASS" and int(r["collected_item_count"] or 0)>0 for r in er),
                     "collection_unavailable":sum(r["collection_status"]=="EVALUATOR_UNAVAILABLE" for r in er),
                     "baseline_passing":sum(r["baseline_status"]=="PASS" for r in er),
                     "baseline_unavailable":sum(r["baseline_status"]=="EVALUATOR_UNAVAILABLE" for r in er)})
    def add_diag(name, x, n=None):
        diagnostics.append({"condition":cond,"diagnostic":name,"numerator":x,"denominator":n,
                            "fraction":x/n if n else None})
    reviewed = [r for r in result["claims"] if r["task_id"] in keys]
    faults = [r for r in result["faults"] if r["task_id"] in keys and exe[r["task_id"]]["baseline_status"]=="PASS"
              and r["status"] not in {"EVALUATOR_UNAVAILABLE","UNATTRIBUTED_OR_RUNTIME_FAILURE"}]
    vars_ = [r for r in result["variations"] if r["task_id"] in keys and r["status"] in {"ACCEPTED","REJECTED_ASSERTION"}]
    requests = [r for r in result["requests"] if r["task_id"] in keys]
    unknown = sum((r.get("classification") or {}).get("classifiable") is not True
                  and r.get("reason_code")!="REQUEST_OUTSIDE_TARGET_OPERATION" for r in requests)
    add_diag("literal_pooled_semantic_supported",sum(r["review_label"]=="supported" for r in reviewed),len(reviewed))
    add_diag("conditional_fault_detection",sum(r["status"]=="KILLED_ASSERTION" for r in faults),len(faults))
    add_diag("legitimate_variation_acceptance",sum(r["status"]=="ACCEPTED" for r in vars_),len(vars_))
    add_diag("unknown_request_intent",unknown,len(requests))
    add_diag("vacuous_baseline_pass",sum(r["baseline_status"]=="PASS" and r["no_request"].lower()=="true" for r in er))
    add_diag("collected_items",sum(int(r["collected_item_count"] or 0) for r in er))
    observed_items = Counter(v for k in keys for v in result["baseline"][k]["item_outcomes"].values())
    for status in ("passed","failed","skipped","xfailed","xpassed","error"):
        add_diag(f"baseline_items_{status}",observed_items[status])
flow = pilot_sources["03"].json("reports/audit/final_eligibility_reconciliation.json")
flow_records = [
    {"milestone":"stage03_input_operations","unit":"operation","count":flow["operation_count"]},
    *({"milestone":f"stage03_first_exit_{k}","unit":"operation","count":v} for k,v in flow["first_exit_status_counts"].items()),
    {"milestone":"stage03_eligible","unit":"specification","count":flow["eligible_specification_count"]},
    {"milestone":"stage03_eligible","unit":"family","count":flow["eligible_family_count"]},
    {"milestone":"stage03_eligible","unit":"operation","count":flow["eligible_operation_count"]},
    {"milestone":"pilot","unit":"operation","count":len(pilot_selection)},
    {"milestone":"main_eligible_remainder","unit":"operation","count":sum(plan["N_h"].values())},
    {"milestone":"main_selected","unit":"operation","count":len(selection)},
    {"milestone":"main_selected","unit":"family","count":len({r["family_id"] for r in selection.values()})},
    {"milestone":"main_selected","unit":"specification","count":len({r["spec_id"] for r in selection.values()})},
]
eligible = pilot_sources["03"].csv("data/inventory/eligible_operations.csv")
edges = plan["frozen_prose_boundaries"]
by_stratum = lambda v: "low" if v<=edges["c1"] else "medium" if v<=edges["c2"] else "high"
balance = []
for name, data in (("stage03_eligible",eligible),("pilot",list(pilot_selection.values())),
                   ("main_eligible_remainder",reserve),("main_selected",list(selection.values()))):
    for h, members in group(data,lambda r:by_stratum(int(r["prose_tokens_L0"]))).items():
        balance.append({"sample":name,"stratum":h,"operations":len(members),
                        "families":len({r["family_id"] for r in members}),
                        "specifications":len({r["spec_id"] for r in members}),
                        "mean_L0_prose_tokens":mean(int(r["prose_tokens_L0"]) for r in members),
                        "native_openapi":sum(r.get("source_class")=="native_openapi" for r in members),
                        "methods_json":json.dumps(Counter(r.get("method","") for r in members),sort_keys=True)})
lines = ["# Main OpenAPI prose experiment · Exploratory report", "",
         f"All {len(selection)} selected operation pairs are accounted for.",
         "Results describe pairs with usable evidence for each outcome. The registered confirmatory run was not completed.",
         "", "## Descriptive paired results", "",
         "| Endpoint | Available | Missing | L0 | L1 | L1 minus L0 |",
         "|---|---:|---:|---:|---:|---:|"]
for row in primary:
    values = [row[k] for k in ("observed_pairs_unweighted_L0", "observed_pairs_unweighted_L1",
                                "observed_pairs_unweighted_L1_minus_L0")]
    labels = " | ".join("Unavailable" if value is None else f"{value:.4f}" for value in values)
    lines.append(f"| {row['endpoint']} | {row['common_evaluable_pairs']} | {row['missing_pairs']} | {labels} |")
lines += ["", "Descriptive values use unweighted means over common-evaluable pairs.",
          "A missing score is retained with its reason. The review dispositions and the technical oracle scope are in the provenance files.",
          "The experiment measures generated tests against documented offline responses."]


In [ ]:
# Notebook 11 · Cell 04 · Export the report, figure and reproducibility package

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9,4.6), layout="constrained")
for i, row in enumerate(primary):
    delta = row["observed_pairs_unweighted_L1_minus_L0"]
    if delta is None:
        ax.text(.01, i, "Unavailable", transform=ax.get_yaxis_transform(), va="center")
    else:
        ax.plot(delta, i, "o", color="#1c5c70")
ax.axvline(0, color="#85959a", linewidth=1)
ax.set_yticks(range(len(primary)), [row["endpoint"].replace("_", " ") for row in primary])
ax.invert_yaxis()
ax.set_xlabel("Mean paired difference (L1 − L0), common-evaluable pairs")
figure = io.BytesIO()
fig.savefig(figure,format="png",dpi=175,metadata={"Software":"OpenAPI-LLM Notebook 11"})
plt.close(fig)
manifest_sources = dict(sources, **{"10":plan_release})
audit = {"inference_complete": False, "protocol_scope": SCOPE, "confirmatory_endpoints": [],
         "pipeline_sections": ["10.6", "11.2", "15.4", "16.1"],
         "runtime":{**package_runtime(),"matplotlib":matplotlib.__version__},
         "pilot_provenance":{s:{"external_sha256":r.external_hash,"internal_sha256":r.internal_hash}
                             for s,r in pilot_sources.items()},
         "main_provenance":{s:{"external_sha256":r.external_hash,"internal_sha256":r.internal_hash}
                             for s,r in manifest_sources.items()},
         "operation_overlap":0,
         "task_endpoint_record_count":len(result["long"]),
         "paired_endpoint_record_count":len(result["paired"]),
         "complete_seven_endpoint_accounting":True,
         "main_evaluator_version":sources["06"].internal["main_evaluator_version"]}
files = {
 "reports/population_flow.csv":csv_bytes(flow_records), "reports/sample_balance.csv":csv_bytes(balance),
 "reports/stage_outcomes.csv":csv_bytes(outcomes), "reports/primary_results.csv":csv_bytes(primary),
 "reports/secondary_diagnostics.csv":csv_bytes(diagnostics), "reports/token_balance.csv":csv_bytes(tokens),
 "reports/review_reliability.csv":sources["08"].read(EVIDENCE+"review_reliability.csv"),
 "reports/main_report.md":("\n".join(lines)+"\n").encode(), "reports/primary_effects.png":figure.getvalue(),
 "reports/reproducibility.json":json_bytes(audit),
 "data/evidence/scores_long.csv":csv_bytes(result["long"]),
 "data/evidence/metric_evaluation_ledger.csv":csv_bytes(result["long"]),
 "data/evidence/paired_scores.csv":csv_bytes(result["paired"]),
 "data/evidence/paired_metric_ledger.csv":csv_bytes(result["paired"]),
 "reports/metric_record_accounting.csv":csv_bytes(metric_accounting),
 "config/main_partial_evidence_qualification.json":partial_raw,
 "config/exploratory_protocol.json":sources["05"].read("config/exploratory_protocol.json"),
 "config/analysis_registered.yaml":plan_release.read("partitions/main/main_analysis_registered.yaml"),
 "config/main_plan.yaml":plan_release.read("partitions/main/main_plan.yaml"),
 "config/metrics.yaml":sources["06"].read("config/metrics.yaml"),
 "config/planning_assumptions.yaml":plan_release.read("config/planning_assumptions.yaml"),
 "config/meaningful_effects.csv":plan_release.read("config/meaningful_effects.csv"),
 "reports/power_qualification.json":plan_release.read("reports/power_qualification.json"),
 "src/statistics.py":plan_release.read("src/statistics.py"),
 "src/scoring_adapter.py":adapter, "src/metrics.py":sources["06"].read("src/metrics.py"),
}
write_release("11","main",files,manifest_sources,
              {"operation_count":len(selection),"inference_complete":audit["inference_complete"],
               "analysis_mode":"EXPLORATORY_DESCRIPTIVE", "protocol_scope":SCOPE})
